## KoBERT 기반 감정 라벨 재태깅

In [2]:
# import


import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm 

from kobert_tokenizer import KoBERTTokenizer

In [3]:
# 1. 모델 및 토크나이저 로드
MODEL_NAME = "skt/kobert-base-v1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = KoBERTTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(8002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)

In [4]:
# 2. 문장 임베딩 함수 ([CLS] 벡터 추출)
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding

In [5]:
# 3. 평균 벡터 계산
def get_average_vector(texts):
    vectors = [get_embedding(t) for t in tqdm(texts)]
    return sum(vectors) / len(vectors)

In [6]:
# 4. 데이터 불러오기
df = pd.read_csv("../../01_kobert_finetuning_prediction/data/Q_review.csv")
df = df.dropna(subset=["구매자평점", "리뷰상세내용"])

In [7]:
# 5. 감정 그룹 나누기 (중립 제거)
positive_reviews = df[df["구매자평점"] >= 4]["리뷰상세내용"].tolist()
negative_reviews = df[df["구매자평점"] <= 2]["리뷰상세내용"].tolist()

In [ ]:
# 6. 평균 임베딩 벡터 생성
print("✅ 긍정 리뷰 임베딩 중...")
pos_vec = get_average_vector(positive_reviews)
print("✅ 부정 리뷰 임베딩 중...")
neg_vec = get_average_vector(negative_reviews)

In [ ]:
# 7. 전체 리뷰 임베딩 및 라벨 재태깅
def relabel(text):
    vec = get_embedding(text).reshape(1, -1)
    sims = {
        "positive": cosine_similarity(vec, pos_vec.reshape(1, -1))[0][0],
        "negative": cosine_similarity(vec, neg_vec.reshape(1, -1))[0][0],
    }
    return max(sims, key=sims.get)

print("✅ 전체 리뷰 재태깅 중...")
df["re_labeled"] = [relabel(text) for text in tqdm(df["리뷰상세내용"])]

In [ ]:
# 8. 저장
df.to_csv("result/re_labeled_Q.csv", index=False)
print("🎉 재태깅 완료! → re_labeled_Q.csv")